# 2. Tokenizer and Local Forward Pass

In the previous notebook, we downloaded and inspected the files that make up `Qwen/Qwen3-0.6B-Base`. We now cross the boundary between artifacts stored on disk and an executable language model. We will first follow text through the tokenizer, then load the pretrained parameters and inspect one forward pass.

The path we want to make explicit is:

```text
text -> tokens -> token IDs -> model -> logits
```

This notebook stops at the next-token distribution. Autoregressive generation will build on this forward pass in the next notebook.

## 1. Reproduce the model selection

We start from a fresh kernel and explicitly select the same model revision inspected in notebook 01. The model ID identifies the repository, while the commit SHA fixes its exact contents even if `main` changes later. No state is imported from the previous notebook.

We also select the device and runtime dtype before loading any tensors. The checkpoint is stored in BF16, and our Apple Silicon environment can preserve that dtype on MPS. This runtime choice is separate from the dtype used to store the checkpoint on disk. Finally, we set and record a seed so later experiments follow the same reproducibility convention.

In [1]:
import json
import platform
import sys
from datetime import datetime, timezone
from importlib.metadata import version

import torch

MODEL_ID = "Qwen/Qwen3-0.6B-Base"
REVISION = "da87bfb608c14b7cf20ba1ce41287e8de496c0cd"
SEED = 123

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    MODEL_DTYPE = torch.bfloat16
else:
    DEVICE = torch.device("cpu")
    MODEL_DTYPE = torch.float32

torch.manual_seed(SEED)

experiment = {
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0],
    "os": platform.platform(),
    "packages": {
        name: version(name)
        for name in ("torch", "transformers", "huggingface-hub", "safetensors")
    },
    "model_id": MODEL_ID,
    "revision": REVISION,
    "device": str(DEVICE),
    "runtime_dtype": str(MODEL_DTYPE),
    "seed": SEED,
}

print(json.dumps(experiment, indent=2))

{
  "started_utc": "2026-10-12T10:06:46.351932+00:00",
  "python": "3.12.13",
  "os": "macOS-26.6.2-arm64-arm-64bit",
  "packages": {
    "torch": "2.11.0",
    "transformers": "5.19.0",
    "huggingface-hub": "2.2.0",
    "safetensors": "0.8.0"
  },
  "model_id": "Qwen/Qwen3-0.6B-Base",
  "revision": "da87bfb608c14b7cf20ba1ce41287e8de496c0cd",
  "device": "mps",
  "runtime_dtype": "torch.bfloat16",
  "seed": 123
}


## 2. Load and inspect the tokenizer

We first load the tokenizer associated with the pinned checkpoint. `local_files_only=True` makes this an explicit cache lookup: the cell will fail instead of silently using the network if any required tokenizer artifact is missing. Loading the tokenizer constructs a text-processing object in CPU memory, but it does not construct the neural network or load `model.safetensors`.

A token is a text piece from the tokenizer vocabulary, while a token ID is the integer position used to select a row from the model's embedding matrix. We inspect both representations for one English sentence and one Spanish sentence. We disable automatic special tokens in this first experiment so the observed IDs come only from the text we supplied, and we decode them again to verify the round trip.

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    local_files_only=True,
)

tokenizer_info = {
    "class": type(tokenizer).__name__,
    "base_vocabulary_size": tokenizer.vocab_size,
    "total_tokens_with_added_tokens": len(tokenizer),
    "model_max_length": tokenizer.model_max_length,
    "model_input_names": tokenizer.model_input_names,
    "special_tokens": tokenizer.special_tokens_map,
    "special_token_ids": dict(zip(tokenizer.all_special_tokens, tokenizer.all_special_ids)),
}
print(json.dumps(tokenizer_info, indent=2, ensure_ascii=False))

examples = [
    "The model predicts the next token.",
    "El modelo predice el siguiente token.",
]

for text in examples:
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    token_pieces = tokenizer.convert_ids_to_tokens(token_ids)
    decoded_text = tokenizer.decode(token_ids, skip_special_tokens=False)

    print(f"\nText:    {text}")
    print(f"Tokens:  {token_pieces}")
    print(f"IDs:     {token_ids}")
    print(f"Decoded: {decoded_text}")

    assert decoded_text == text

{
  "class": "Qwen2Tokenizer",
  "base_vocabulary_size": 151643,
  "total_tokens_with_added_tokens": 151669,
  "model_max_length": 131072,
  "model_input_names": [
    "input_ids",
    "attention_mask"
  ],
  "special_tokens": {
    "eos_token": "<|endoftext|>",
    "pad_token": "<|endoftext|>"
  },
  "special_token_ids": {
    "<|endoftext|>": 151643,
    "<|im_start|>": 151644,
    "<|im_end|>": 151645,
    "<|object_ref_start|>": 151646,
    "<|object_ref_end|>": 151647,
    "<|box_start|>": 151648,
    "<|box_end|>": 151649,
    "<|quad_start|>": 151650,
    "<|quad_end|>": 151651,
    "<|vision_start|>": 151652,
    "<|vision_end|>": 151653,
    "<|vision_pad|>": 151654,
    "<|image_pad|>": 151655,
    "<|video_pad|>": 151656
  }
}

Text:    The model predicts the next token.
Tokens:  ['The', 'Ġmodel', 'Ġpredicts', 'Ġthe', 'Ġnext', 'Ġtoken', '.']
IDs:     [785, 1614, 55878, 279, 1790, 3950, 13]
Decoded: The model predicts the next token.

Text:    El modelo predice el siguiente t

## 3. Build model inputs

We now ask the tokenizer to return PyTorch tensors. Even a single sequence includes a batch dimension, so both `input_ids` and `attention_mask` have shape `[B, T]`: `B` is the number of sequences and `T` is the number of token positions after padding. The IDs are integer indices into the embedding matrix and remain on CPU until we explicitly move them to the model device.

We then combine sequences of different lengths in one batch. Right padding makes them rectangular by appending the pad token to the shorter sequence. Qwen uses `<|endoftext|>` as both its end-of-text token and its pad token, so the ID alone cannot tell us whether a position contains content or padding; `attention_mask` provides that distinction with `1` for real positions and `0` for padding.

This padding mask is not the causal mask. The padding mask differs across batch items and hides artificial padding positions. The causal mask is triangular and prevents every token from attending to future positions. The model will combine both constraints internally during the forward pass.

In [3]:
single_text = examples[0]
single_inputs = tokenizer(
    single_text,
    add_special_tokens=False,
    return_tensors="pt",
)

print("Single sequence")
for name, tensor in single_inputs.items():
    print(f"{name:15} shape={tuple(tensor.shape)}, dtype={tensor.dtype}, device={tensor.device}")
    print(tensor)

assert single_inputs["input_ids"].shape == single_inputs["attention_mask"].shape
assert torch.all(single_inputs["attention_mask"] == 1)

tokenizer.padding_side = "right"
batch_texts = [single_text, "Hola."]
padded_inputs = tokenizer(
    batch_texts,
    add_special_tokens=False,
    padding=True,
    return_tensors="pt",
)

print("\nPadded batch")
for name, tensor in padded_inputs.items():
    print(f"{name:15} shape={tuple(tensor.shape)}, dtype={tensor.dtype}, device={tensor.device}")
    print(tensor)

for row, text in enumerate(batch_texts):
    ids = padded_inputs["input_ids"][row].tolist()
    pieces = tokenizer.convert_ids_to_tokens(ids)
    print(f"\n{text!r}")
    print("tokens:        ", pieces)
    print("attention_mask:", padded_inputs["attention_mask"][row].tolist())

sequence_length = padded_inputs["input_ids"].shape[1]
causal_visibility = torch.tril(
    torch.ones((sequence_length, sequence_length), dtype=torch.int64)
)
print("\nCausal visibility (rows=query positions, columns=key positions):")
print(causal_visibility)

assert padded_inputs["input_ids"].shape == padded_inputs["attention_mask"].shape
assert torch.any(padded_inputs["attention_mask"] == 0)

Single sequence
input_ids       shape=(1, 7), dtype=torch.int64, device=cpu
tensor([[  785,  1614, 55878,   279,  1790,  3950,    13]])
attention_mask  shape=(1, 7), dtype=torch.int64, device=cpu
tensor([[1, 1, 1, 1, 1, 1, 1]])

Padded batch
input_ids       shape=(2, 7), dtype=torch.int64, device=cpu
tensor([[   785,   1614,  55878,    279,   1790,   3950,     13],
        [ 68012,     13, 151643, 151643, 151643, 151643, 151643]])
attention_mask  shape=(2, 7), dtype=torch.int64, device=cpu
tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 0, 0, 0, 0, 0]])

'The model predicts the next token.'
tokens:         ['The', 'Ġmodel', 'Ġpredicts', 'Ġthe', 'Ġnext', 'Ġtoken', '.']
attention_mask: [1, 1, 1, 1, 1, 1, 1]

'Hola.'
tokens:         ['Hola', '.', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>']
attention_mask: [1, 1, 0, 0, 0, 0, 0]

Causal visibility (rows=query positions, columns=key positions):
tensor([[1, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0

## 4. Load the base model locally

We now turn the cached artifacts into an executable PyTorch model. `AutoModelForCausalLM` reads `config.json`, selects the Qwen3 causal-language-model implementation, constructs its modules and fills their parameters from `model.safetensors`. `local_files_only=True` keeps the operation offline.

The model is created in CPU memory and then moved to our selected device and runtime dtype. Calling `eval()` switches modules to evaluation behavior, but it does not disable gradient tracking; we will use `torch.inference_mode()` around the forward pass later. We inspect the unique parameter count, parameter storage and logical device after the move. On Apple Silicon, CPU and MPS share physical memory, so their memory measurements should not be added as if they were independent pools.

Parameter storage is only the model's baseline memory cost. A forward pass also creates activations, attention intermediates and runtime workspaces, while autoregressive generation adds a growing KV cache.

In [5]:
from collections import Counter

from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    local_files_only=True,
    dtype=MODEL_DTYPE,
)
model = model.to(DEVICE)
model.eval()

parameter_count = sum(parameter.numel() for parameter in model.parameters())
parameter_bytes = sum(
    parameter.numel() * parameter.element_size()
    for parameter in model.parameters()
)
parameter_dtypes = Counter(str(parameter.dtype) for parameter in model.parameters())
parameter_devices = Counter(str(parameter.device) for parameter in model.parameters())

model_info = {
    "class": type(model).__name__,
    "training_mode": model.training,
    "parameters": parameter_count,
    "parameter_storage_GiB": parameter_bytes / 1024**3,
    "parameter_dtypes": dict(parameter_dtypes),
    "parameter_devices": dict(parameter_devices),
    "input_output_embeddings_tied": (
        model.get_input_embeddings().weight is model.get_output_embeddings().weight
    ),
}
if DEVICE.type == "mps":
    model_info["mps_current_allocated_GiB"] = torch.mps.current_allocated_memory() / 1024**3
elif DEVICE.type == "cuda":
    model_info["cuda_current_allocated_GiB"] = torch.cuda.memory_allocated() / 1024**3

print(json.dumps(model_info, indent=2))
print("\nTop-level modules:")
for name, module in model.named_children():
    print(f"{name:10} {type(module).__name__}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

{
  "class": "Qwen3ForCausalLM",
  "training_mode": false,
  "parameters": 596049920,
  "parameter_storage_GiB": 1.1102294921875,
  "parameter_dtypes": {
    "torch.bfloat16": 310
  },
  "parameter_devices": {
    "mps:0": 310
  },
  "input_output_embeddings_tied": true,
  "mps_current_allocated_GiB": 1.4000248908996582
}

Top-level modules:
model      Qwen3Model
lm_head    Linear


## 5. Map Qwen components to mini GPT

Qwen follows the same decoder-only Transformer skeleton as our mini GPT, but replaces several components with modern variants. We inspect one of its 28 repeated decoder layers because every layer has the same module structure while keeping its own learned parameters.

| Role | Mini GPT | Qwen3 |
| --- | --- | --- |
| Token embeddings | `token_embedding` | `model.embed_tokens` |
| Position information | Learned `positional_embedding` added to token embeddings | RoPE applied to queries and keys |
| Normalization | LayerNorm with scale and shift | RMSNorm with scale only |
| Attention projections | Equal numbers of Q, K and V heads | GQA with 16 Q heads and 8 KV heads |
| Query/key normalization | None | RMSNorm inside each Q and K head |
| Feed-forward network | Linear -> GELU -> Linear | Gated SwiGLU-style MLP |
| Output projection | Independent `output_head` | `lm_head` tied to input embeddings |

Qwen's MLP computes `down_proj(silu(gate_proj(x)) * up_proj(x))`. The element-wise product creates a learned gate before projecting back to the residual-stream width. Its RMSNorm divides by the root mean square without subtracting the mean, unlike the LayerNorm implemented in mini GPT. RoPE encodes position by rotating pairs of query and key features instead of adding a learned position vector to the token embedding.

The attention projections expose GQA directly in their shapes. Queries use `16 * 128 = 2048` output features, while keys and values use `8 * 128 = 1024`. Each KV head is therefore shared by two query heads. After attention, `o_proj` maps the concatenated 2048 query-head features back to the 1024-dimensional residual stream.

In [6]:
backbone = model.model
first_block = backbone.layers[0]
attention = first_block.self_attn
mlp = first_block.mlp

architecture = {
    "token_embedding": {
        "class": type(backbone.embed_tokens).__name__,
        "weight_shape": list(backbone.embed_tokens.weight.shape),
    },
    "decoder_layers": len(backbone.layers),
    "rotary_embedding": type(backbone.rotary_emb).__name__,
    "first_decoder_layer": {
        "class": type(first_block).__name__,
        "input_norm": type(first_block.input_layernorm).__name__,
        "post_attention_norm": type(first_block.post_attention_layernorm).__name__,
        "attention": {
            "class": type(attention).__name__,
            "is_causal": attention.is_causal,
            "query_heads": model.config.num_attention_heads,
            "key_value_heads": model.config.num_key_value_heads,
            "key_value_groups": attention.num_key_value_groups,
            "head_dim": attention.head_dim,
            "scale": attention.scaling,
            "q_proj": list(attention.q_proj.weight.shape),
            "k_proj": list(attention.k_proj.weight.shape),
            "v_proj": list(attention.v_proj.weight.shape),
            "o_proj": list(attention.o_proj.weight.shape),
            "q_norm": type(attention.q_norm).__name__,
            "k_norm": type(attention.k_norm).__name__,
        },
        "mlp": {
            "class": type(mlp).__name__,
            "activation": type(mlp.act_fn).__name__,
            "gate_proj": list(mlp.gate_proj.weight.shape),
            "up_proj": list(mlp.up_proj.weight.shape),
            "down_proj": list(mlp.down_proj.weight.shape),
        },
    },
    "final_norm": type(backbone.norm).__name__,
    "language_model_head": {
        "class": type(model.lm_head).__name__,
        "weight_shape": list(model.lm_head.weight.shape),
    },
}

print(json.dumps(architecture, indent=2))

assert attention.q_proj.out_features == model.config.num_attention_heads * attention.head_dim
assert attention.k_proj.out_features == model.config.num_key_value_heads * attention.head_dim
assert attention.v_proj.out_features == model.config.num_key_value_heads * attention.head_dim
assert attention.o_proj.out_features == model.config.hidden_size

{
  "token_embedding": {
    "class": "Embedding",
    "weight_shape": [
      151936,
      1024
    ]
  },
  "decoder_layers": 28,
  "rotary_embedding": "Qwen3RotaryEmbedding",
  "first_decoder_layer": {
    "class": "Qwen3DecoderLayer",
    "input_norm": "Qwen3RMSNorm",
    "post_attention_norm": "Qwen3RMSNorm",
    "attention": {
      "class": "Qwen3Attention",
      "is_causal": true,
      "query_heads": 16,
      "key_value_heads": 8,
      "key_value_groups": 2,
      "head_dim": 128,
      "scale": 0.08838834764831845,
      "q_proj": [
        2048,
        1024
      ],
      "k_proj": [
        1024,
        1024
      ],
      "v_proj": [
        1024,
        1024
      ],
      "o_proj": [
        1024,
        2048
      ],
      "q_norm": "Qwen3RMSNorm",
      "k_norm": "Qwen3RMSNorm"
    },
    "mlp": {
      "class": "Qwen3MLP",
      "activation": "SiLUActivation",
      "gate_proj": [
        3072,
        1024
      ],
      "up_proj": [
        3072,
        102

## 6. Run one forward pass

We now move the unpadded input tensors from CPU to the same device as the model and execute one forward pass. `model.eval()` has already selected evaluation behavior, while `torch.inference_mode()` prevents PyTorch from building an autograd graph. We also disable the KV cache because this experiment processes one complete sequence only; caching becomes useful when autoregressive generation repeatedly extends the same prefix.

Transformers returns a structured output rather than the logits tensor directly. For a causal language model, `outputs.logits` has shape `[B, T, V]`: one batch item, one output position for every input token, and one unnormalized score for every vocabulary entry. `logits[b, t, v]` scores token ID `v` as the token following the prefix ending at position `t`. The forward pass does not apply softmax and does not select a token.

In [7]:
forward_inputs = {
    name: tensor.to(DEVICE)
    for name, tensor in single_inputs.items()
}

with torch.inference_mode():
    outputs = model(
        **forward_inputs,
        use_cache=False,
    )

logits = outputs.logits
batch_size, sequence_length, vocabulary_size = logits.shape

forward_record = {
    "output_class": type(outputs).__name__,
    "output_fields": list(outputs.keys()),
    "input_ids_shape": list(forward_inputs["input_ids"].shape),
    "input_device": str(forward_inputs["input_ids"].device),
    "logits_shape": list(logits.shape),
    "logits_dtype": str(logits.dtype),
    "logits_device": str(logits.device),
    "batch_size_B": batch_size,
    "sequence_length_T": sequence_length,
    "vocabulary_size_V": vocabulary_size,
}
print(json.dumps(forward_record, indent=2))

assert forward_inputs["input_ids"].device.type == DEVICE.type
assert logits.shape[:2] == forward_inputs["input_ids"].shape
assert vocabulary_size == model.config.vocab_size

{
  "output_class": "CausalLMOutputWithPast",
  "output_fields": [
    "logits"
  ],
  "input_ids_shape": [
    1,
    7
  ],
  "input_device": "mps:0",
  "logits_shape": [
    1,
    7,
    151936
  ],
  "logits_dtype": "torch.bfloat16",
  "logits_device": "mps:0",
  "batch_size_B": 1,
  "sequence_length_T": 7,
  "vocabulary_size_V": 151936
}


## 7. Inspect the next-token distribution

We select `logits[:, -1, :]` because our input contains one unpadded sequence and its final column corresponds to the period at the end of the complete prompt. This reduces `[B, T, V]` to `[B, V]`. After converting the BF16 logits to FP32, softmax turns their relative scores into a normalized next-token distribution.

We inspect the ten most probable candidates and use argmax as a deterministic example. The raw tokenizer piece preserves internal markers such as `Ġ`, while decoding the same ID produces the exact text fragment that would be appended. This selects only one continuation token; autoregressive generation will repeat this process in the next notebook.

Selecting column `-1` is not generally correct for a right-padded batch because the final columns of shorter sequences are padding. In that case, we must locate each sequence's last real position using its attention mask, or use left padding so every sequence ends at the final column.

In [8]:
next_token_logits = logits[:, -1, :].float()
next_token_probabilities = torch.softmax(next_token_logits, dim=-1)

top_k = 10
top_probabilities, top_token_ids = torch.topk(
    next_token_probabilities,
    k=top_k,
    dim=-1,
)

top_probabilities = top_probabilities[0].cpu().tolist()
top_token_ids = top_token_ids[0].cpu().tolist()

print(f"Prompt: {single_text!r}")
print(f"Next-token distribution shape: {tuple(next_token_probabilities.shape)}")
print(f"\n{'Rank':>4} {'Token ID':>9} {'Probability':>12}  {'Raw piece':<18} Decoded piece")
for rank, (token_id, probability) in enumerate(
    zip(top_token_ids, top_probabilities),
    start=1,
):
    raw_piece = tokenizer.convert_ids_to_tokens(token_id)
    decoded_piece = tokenizer.decode([token_id], skip_special_tokens=False)
    print(
        f"{rank:>4} {token_id:>9} {probability:>11.4%}  "
        f"{raw_piece!r:<18} {decoded_piece!r}"
    )

greedy_token_id = top_token_ids[0]
greedy_text_piece = tokenizer.decode([greedy_token_id], skip_special_tokens=False)
print(f"\nGreedy token ID: {greedy_token_id}")
print(f"Greedy continuation: {single_text + greedy_text_piece!r}")

Prompt: 'The model predicts the next token.'
Next-token distribution shape: (1, 151936)

Rank  Token ID  Probability  Raw piece          Decoded piece
   1       576    15.9956%  'ĠThe'             ' The'
   2      1096     4.5828%  'ĠThis'            ' This'
   3       758     4.0443%  'ĠIn'              ' In'
   4      1446     3.5691%  'ĠYou'             ' You'
   5      1752     3.5691%  'ĠFor'             ' For'
   6       220     3.1497%  'Ġ'                ' '
   7      1084     3.1497%  'ĠIt'              ' It'
   8      1416     3.1497%  'ĠIf'              ' If'
   9       198     2.7796%  'Ċ'                '\n'
  10      4354     2.4530%  'ĠHowever'         ' However'

Greedy token ID: 576
Greedy continuation: 'The model predicts the next token. The'


## 8. Verify the forward pass

We finish by checking the properties that should hold independently of the model's preferred continuation. The logits must have the expected `[B, T, V]` shape and contain only finite values, while each softmax distribution must sum to one. The tensors produced inside `torch.inference_mode()` must not require gradients.

We also repeat the same forward pass while the model remains in evaluation mode. BF16 has a machine epsilon of approximately `0.0078125`, so we use that value as both the relative and absolute tolerance instead of assuming exact equality across every accelerator kernel. This checks repeatability at the numerical precision used by the experiment.

In [9]:
expected_logits_shape = (
    forward_inputs["input_ids"].shape[0],
    forward_inputs["input_ids"].shape[1],
    model.config.vocab_size,
)
all_logits_finite = bool(torch.isfinite(logits).all().item())
probability_sums = next_token_probabilities.sum(dim=-1)
maximum_probability_sum_error = float(
    (probability_sums - 1.0).abs().max().item()
)

with torch.inference_mode():
    repeated_outputs = model(
        **forward_inputs,
        use_cache=False,
    )
repeated_logits = repeated_outputs.logits

bf16_tolerance = torch.finfo(torch.bfloat16).eps
maximum_repeat_difference = float(
    (logits.float() - repeated_logits.float()).abs().max().item()
)
repeated_forward_is_close = torch.allclose(
    logits.float(),
    repeated_logits.float(),
    rtol=bf16_tolerance,
    atol=bf16_tolerance,
)

checks = {
    "expected_logits_shape": list(expected_logits_shape),
    "actual_logits_shape": list(logits.shape),
    "all_logits_finite": all_logits_finite,
    "logits_require_grad": logits.requires_grad,
    "probability_sum": float(probability_sums[0].item()),
    "maximum_probability_sum_error": maximum_probability_sum_error,
    "repeat_tolerance": bf16_tolerance,
    "maximum_repeat_difference": maximum_repeat_difference,
    "repeated_forward_is_close": repeated_forward_is_close,
    "greedy_token_id": greedy_token_id,
    "greedy_text_piece": greedy_text_piece,
}
print(json.dumps(checks, indent=2, ensure_ascii=False))

assert logits.shape == expected_logits_shape
assert all_logits_finite
assert not logits.requires_grad
torch.testing.assert_close(
    probability_sums,
    torch.ones_like(probability_sums),
    rtol=0.0,
    atol=1e-6,
)
assert repeated_forward_is_close

{
  "expected_logits_shape": [
    1,
    7,
    151936
  ],
  "actual_logits_shape": [
    1,
    7,
    151936
  ],
  "all_logits_finite": true,
  "logits_require_grad": false,
  "probability_sum": 1.000000238418579,
  "maximum_probability_sum_error": 2.384185791015625e-07,
  "repeat_tolerance": 0.0078125,
  "maximum_repeat_difference": 0.0,
  "repeated_forward_is_close": true,
  "greedy_token_id": 576,
  "greedy_text_piece": " The"
}


## Conclusions

We loaded the tokenizer and the pinned `Qwen/Qwen3-0.6B-Base` checkpoint entirely from the local Hugging Face cache. The tokenizer converted English and Spanish text into model-specific token IDs, and the padding experiment separated the `[B, T]` padding mask from the triangular causal constraint. We then constructed `Qwen3ForCausalLM`, moved its 596,049,920 BF16 parameters to MPS and connected its decoder-only structure to mini GPT. The overall path remains embeddings, causal self-attention, an MLP and an output projection, while Qwen replaces learned positional embeddings, LayerNorm, standard multi-head attention and GELU with RoPE, RMSNorm, GQA, Q/K normalization and a gated SiLU MLP.

A forward pass over seven input tokens produced finite BF16 logits with shape `[1, 7, 151936]`. Selecting the final position and applying softmax in FP32 produced a normalized next-token distribution; greedy decoding selected token ID `576`, which decodes to `" The"` with a probability of approximately 16%. This was a single local forward pass rather than generation: the next notebook will repeatedly select a token, append it to the sequence and reuse the prefix efficiently with a KV cache.